In [ ]:
## SRSK

## nitin_t.iitr@paramganga.iitr.ac.in
## nitin_t.iitr@@321
## /scratch/nitin_t.iitr/

## shitiz_g_mfs.iitr@paramganga.iitr.ac.in
## srsk@1234
## /scratch/shitiz_g_mfs.iitr/



## ---- Importing libraries ---- ##

import os
import glob
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary

from sklearn.metrics import classification_report, confusion_matrix

import spectral as spy
import spectral.io.envi as envi

import matplotlib.pyplot as plt




## ---- Constants ---- ##

batch_size = 64
input_shape = (56, 56, 147)
num_classes = 4
eta = 0.001                 # Initial Learning Rate
weight_decay = 0.01        # L2 regularizatiion
optimizer_type = 'sgd'      # 'adam' or 'sgd'
epochs = 100
pg = 'ns'

if pg == 'sg':
    train_dir_path =    '/scratch/shitiz_g_mfs.iitr/Dataset/Train'
    val_dir_path =      '/scratch/shitiz_g_mfs.iitr/Dataset/Validation'
    test_dir_path =     '/scratch/shitiz_g_mfs.iitr/Dataset/Test'

    model_save_path = '/scratch/shitiz_g_mfs.iitr/Models/ResNet3D/resnet34_3d_01_final.pth'
    best_model_path = '/scratch/shitiz_g_mfs.iitr/Models/ResNet3D/resnet34_3d_01_best.pth'
    csv_path =        '/scratch/shitiz_g_mfs.iitr/Models/ResNet3D/resnet34_3d_01.csv'
elif pg == 'ns':
    train_dir_path =    '/scratch/nitin_t.iitr/Nitin_Sir_Data/50_varieties_data/Train'
    val_dir_path =      '/scratch/nitin_t.iitr/Nitin_Sir_Data/50_varieties_data/Validation'
    test_dir_path =     '/scratch/nitin_t.iitr/Nitin_Sir_Data/50_varieties_data/Test'

    model_save_path = '/scratch/nitin_t.iitr/Nitin_Sir_Data/Models/ResNet3D/resnet34_3d_01_final.pth'
    best_model_path = '/scratch/nitin_t.iitr/Nitin_Sir_Data/Models/ResNet3D/resnet34_3d_01_best.pth'
    csv_path =        '/scratch/nitin_t.iitr/Nitin_Sir_Data/Models/ResNet3D/resnet34_3d_01.csv'
    cfm_path =        '/scratch/nitin_t.iitr/Nitin_Sir_Data/Models/ResNet3D/resnet34_3d_01_cfm.csv'


print("Batch Size : ", batch_size)
print("Number of classes : ", num_classes)
print("Optimizer : ", optimizer_type)
print("Initial learning rate : ", eta)
print("Weight decay alpha : ", weight_decay)




## ---- Data Loading ---- ##

class WheatDataset(Dataset):
    def __init__(self, directory, transform=None):
        self.directory = directory
        self.transform = transform
        self.classes = sorted(os.listdir(directory))
        self.class_indices = {c: i for i, c in enumerate(self.classes)}
        self.file_paths = []
        self.labels = []

        # Used to create self.filepaths and self.labels lists
        for class_name in self.classes:
            class_path = os.path.join(directory, class_name)
            files = glob.glob(os.path.join(class_path, '*.img'))
            label = self.class_indices[class_name]
            self.file_paths.extend(files)
            self.labels.extend([label] * len(files))

    def __len__(self):
        return len(self.file_paths)

    # Loading and Normalizating the Images
    def load_image(self, file_path):                    # Used in __getitem__ function
        hdr_path = file_path.replace('.img', '.hdr')
        img = envi.open(hdr_path, file_path).load().astype(np.float32)
        img_min = np.min(img)
        img_max = np.max(img)
        img = (img - img_min) / (img_max - img_min)
        img = np.expand_dims(img, axis=0)  # Shape: (1, Depth, Height, Width) Extra for 3D CNN

        return img

    def __getitem__(self, idx):
        img = self.load_image(self.file_paths[idx])
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label


# Transforming to tensor as well as resizing
transform = transforms.Compose([
    transforms.Lambda(lambda x: torch.tensor(x, dtype=torch.float32)),  # Converts numpy array to tensor
    #transforms.Resize((224, 224)),
])


train_dataset = WheatDataset(train_dir_path, transform=transform)
val_dataset = WheatDataset(val_dir_path, transform=transform)
test_dataset = WheatDataset(test_dir_path, transform=transform)

# Shuffling the Training Dataset
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in val_loader: {len(val_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

print(f"\nNumber of samples in train_dataset: {len(train_loader.dataset)}")
print(f"Number of samples in val_dataset: {len(val_loader.dataset)}")
print(f"Number of samples in test_dataset: {len(test_loader.dataset)}")




## ---- Model Implementation ---- ##

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):

    def __init__(self, block, layers, channels, num_classes=50):
        super().__init__()

        self.inplanes = 64
        #self.conv1 = nn.Conv3d(1, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.conv1 = nn.Conv3d(1, self.inplanes, kernel_size=3, stride=(2,1,1), padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=3, stride=2, padding=1)

        # channels = [16, 32, 64, 128], layers=[3, 4, 6, 3]
        self.layer1 = self._make_layer(block, channels[0], layers[0])
        self.layer2 = self._make_layer(block, channels[1], layers[1], stride=2)
        self.layer3 = self._make_layer(block, channels[2], layers[2], stride=2)
        self.layer4 = self._make_layer(block, channels[3], layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(channels[3] , num_classes)


    def _make_layer(self, block, planes, noofblocks, stride=1):
        downsample = None

        if stride != 1 or self.inplanes != planes:  # 1x1 Conv(S=2,P=0) + BN
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, planes, kernel_size=1, stride, bias=False),
                nn.BatchNorm3d(planes),
            )

        layers = []
        # Stride 2 for first layer only
        layers.append(block(self.inplanes, planes, stride, downsample))

        self.inplanes = planes
        for _ in range(1, noofblocks):
            layers.append(block(self.inplanes, planes)) # Stride 1 for subsequent blocks

        return nn.Sequential(*layers)


    def forward(self, x):
        x = self.conv1(x)           # 224x224
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)         # 112x112

        x = self.layer1(x)          # 56x56
        x = self.layer2(x)          # 28x28
        x = self.layer3(x)          # 14x14
        x = self.layer4(x)          # 7x7

        x = self.avgpool(x)         # 1x1
        x = torch.flatten(x, 1)     # remove 1 X 1 grid and make vector of tensor shape
        x = self.fc(x)

        return x

def resnet34():
    layers=[3, 4, 6, 3]
    channels = [16, 32, 64, 128]    # Original [64, 128, 256, 512]
    model = ResNet(BasicBlock, layers, channels)
    return model

model=resnet34()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available Device :" , device)

model = model.to(device)
print("\n", model)

print("\nSummary of the model:\n")
print(summary(model, (1,147,56,56)))




## ---- Training ---- ##

# Optimizer
if optimizer_type == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=eta, weight_decay=weight_decay)
elif optimizer_type == 'sgd':
    optimizer = optim.SGD(model.parameters(), lr=eta, momentum=0.9, nesterov=True, weight_decay=weight_decay)

# Loss Function
criterion = nn.CrossEntropyLoss()

# LR Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                       mode='min', factor=0.1,
                                                       patience=5, verbose=True,
                                                       min_lr=1e-6)

class training():
    def __init__(self, device, model, train_loader, val_loader, criterion, optimizer, epochs, save_path, scheduler = None):
        self.device = device
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.epochs = epochs
        self.save_path = save_path

        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.epochs_list = []


    def evaluate_model(self):
        print("Evaluating Model on Validation Data")
        # Set the model to evaluate mode
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in self.val_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        loss = running_loss / len(self.val_loader.dataset)
        acc = correct / total
        return loss, acc


    def train_model(self):
        print("\n----- Model Training Started -----")
        #best_val_acc = 0.0              # Initialize the best validation accuracy

        best_val_loss = float('inf')
        patience = 10  # Number of epochs to wait for improvement
        counter = 0  # Counts epochs with no improvement

        train_start_time = time.time()
        for epoch in range(self.epochs):
            start_time = time.time()  # Start time for the epoch

            # Set the model to training mode - important for batch normalization and dropout layers
            self.model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            print(f"Epoch {epoch+1} started.")
            for inputs, labels in self.train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(inputs)             # Forward pass
                loss = self.criterion(outputs, labels)   # Compute loss
                loss.backward()                          # Backward pass
                self.optimizer.step()                    # Update weights

                running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

            epoch_loss = running_loss / len(self.train_loader.dataset)
            epoch_acc = correct / total

            val_loss, val_acc = self.evaluate_model()
            if self.scheduler:
                self.scheduler.step(val_loss)

            self.epochs_list.append(epoch + 1)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            self.train_losses.append(epoch_loss)
            self.train_accuracies.append(epoch_acc)

            end_time = time.time()  # End time for the epoch
            epoch_time = end_time - start_time  # Calculate the time taken for the epoch

            print(f'Epoch {epoch+1}/{self.epochs} - [Time: {epoch_time:.2f} seconds]  Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, '
                f'Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}, LR: {self.optimizer.param_groups[0]["lr"]:.6f}')

            # Early Stopping
            # After each epoch, check the validation loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                counter = 0  # Reset the counter if there's improvement
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping at epoch - {epoch+1}")
                    break  # Exit the training loop

            # # Save the model with the best validation accuracy
            # if val_acc > best_val_acc:
            #     best_val_acc = val_acc
            #     torch.save({
            #         'model_state_dict': self.model.state_dict(),
            #         'optimizer_state_dict': self.optimizer.state_dict(),
            #         'epoch': epoch,
            #         'val_accuracy': best_val_acc,
            #     }, self.save_path)

        train_end_time = time.time()
        total_time = train_end_time - train_start_time
        print(f"Total time taken - {total_time/60 : .2f} minutes")


    def get_parameters(self):
        return self.train_losses, self.train_accuracies, self.val_losses, self.val_accuracies


    def save_parameters_csv(self, csv_path):
        data = {
            'epoch': self.epochs_list,
            'train_loss': self.train_losses,
            'train_accuracy': self.train_accuracies,
            'val_loss': self.val_losses,
            'val_accuracy': self.val_accuracies
        }

        # Convert the dictionary into a DataFrame
        df = pd.DataFrame(data)

        # Save the DataFrame to a CSV file
        df.to_csv(csv_path, index=False)


    def plot_learning_curve(self):
        # Plot Loss
        plt.figure(figsize=(10, 5))
        plt.plot(self.train_losses, label='Training Loss')
        plt.plot(self.val_losses, label='Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Learning Curve - Loss')
        plt.legend()
        plt.show()

        # Plot Accuracy
        plt.figure(figsize=(10, 5))
        plt.plot(self.train_accuracies, label='Training Accuracy')
        plt.plot(self.val_accuracies, label='Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.title('Learning Curve - Accuracy')
        plt.legend()
        plt.show()



train = training(device, model, train_loader, val_loader, criterion, optimizer, epochs, best_model_path, scheduler)
train.train_model()
# train_losses, train_accuracies, val_losses, val_accuracies = train.get_parameters()
# train.plot_learning_curve()
train.save_parameters_csv(csv_path)




## ---- Model Testing ---- ##

# Free up memory
del optimizer               # If optimizer is no longer needed
torch.cuda.empty_cache()    # Clear any cached memory on GPU

class testing():
    def __init__(self, directory, device, model, test_loader, criterion):
        self.directory = directory
        self.class_names = sorted(os.listdir(directory))
        self.device = device
        self.model = model
        self.test_loader = test_loader
        self.criterion = criterion


    def test_model(self, cfm_path):
        print("\n----- Model Testing Started -----")

        # Set the model to evaluation mode (turns off dropout and batch norm layers)
        self.model.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        all_labels = []
        all_preds = []

        with torch.no_grad():  # No need to calculate gradients during testing
            for inputs, labels in self.test_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                outputs = self.model(inputs)                 # Forward pass
                loss = self.criterion(outputs, labels)       # Compute loss

                running_loss += loss.item() * inputs.size(0) # Accumulate test loss
                _, predicted = outputs.max(1)                # Get the index of the max log-probability

                total += labels.size(0)
                correct += predicted.eq(labels).sum().item() # Count correct predictions

                # Store labels and predictions for classification report
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())

        test_loss = running_loss / len(self.test_loader.dataset)
        test_acc = correct / total

        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

        # Generate classification report
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=self.class_names))

        # Generate confusion matrix
        print("\nConfusion Matrix:")
        conf_matrix = confusion_matrix(all_labels, all_preds)
        df_conf_matrix = pd.DataFrame(conf_matrix, index=self.class_names, columns=self.class_names)
        print(df_conf_matrix)
        df_conf_matrix.to_csv(cfm_path, index=False)

# After training completes
test = testing(test_dir_path, device, model, test_loader, criterion)
test.test_model(cfm_path)




## ---- Model Saving ---- ##

def save_model(model, model_save_path):
    print("\nFinal Model Saving")
    torch.save(model.state_dict(), model_save_path)

save_model(model, model_save_path)

Model: "ResNet34"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 56, 56, 152)]        0         []                            
                                                                                                  
 resizing (Resizing)         (None, 224, 224, 152)        0         ['input_1[0][0]']             
                                                                                                  
 conv2d (Conv2D)             (None, 112, 112, 64)         476736    ['resizing[0][0]']            
                                                                                                  
 batch_normalization (Batch  (None, 112, 112, 64)         256       ['conv2d[0][0]']              
 Normalization)                                                                            